In [6]:
!pip install pandas sqlalchemy

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

In [2]:

def build_dim_date_df(date_strs):
    """Build dim_date rows from an iterable of 'YYYY-MM-DD' strings."""
    dates = pd.to_datetime(pd.Series(list(set(date_strs))).dropna()).drop_duplicates()
    df = pd.DataFrame({"date_id": dates.dt.strftime("%Y-%m-%d")})
    df["year"]       = dates.dt.year.values
    df["quarter"]    = dates.dt.quarter.values
    df["month"]      = dates.dt.month.values
    df["month_name"] = dates.dt.strftime("%B").values
    df["week"]       = dates.dt.isocalendar().week.values
    df["day"]        = dates.dt.day.values
    df["day_name"]   = dates.dt.strftime("%A").values
    # is_month_end: last day of month
    df["is_month_end"] = (dates + pd.offsets.MonthEnd(0) == dates).astype(int).values
    return df.drop_duplicates("date_id")
 

In [3]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///bluestock_mf.db"
)

print("Database Created Successfully")

Database Created Successfully


In [5]:



fund_master   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_01_fund_master.csv")
nav_history   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\Cleaned_02_nav_history.csv")
aum_fh        = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_03_aum_by_fund_house.csv")
sip_inflows   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_04_monthly_sip_inflows.csv")
cat_inflows   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_05_category_inflows.csv")
folio_count   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_06_industry_folio_count.csv")
scheme_perf   = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_07_scheme_performance.csv")
transactions  = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_08_investor_transactions.csv")
holdings      = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_09_portfolio_holdings.csv")
benchmarks    = pd.read_csv(r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\processed\cleaned_10_benchmark_indices.csv")
 
print("All cleaned csv file loaded into dataframes ")

All cleaned csv file loaded into dataframes 


In [6]:
print(fund_master.head())

   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 250 SmallCap TRI   
3   Equity    Small Cap   Direct  2013-01-01       BSE 250 SmallCap TRI   
4     Debt         Gilt  Regular  2000-12-30  CRISIL Dynamic Gilt Index   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \

In [29]:
### Load dataframes to SQLITE


fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav_history.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

aum_fh.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)

transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

scheme_perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)


sip_inflows.to_sql(
    "SIP_inflows",
    engine,
    if_exists="replace",
    index=False
)
print("All tables loaded successfully")




All tables loaded successfully


In [30]:
##  Verify Row Counts Match CSVs

tables = {
    "dim_fund": fund_master,
    "fact_nav": nav_history,
    "fact_aum": aum_fh,
    "fact_transactions": transactions,
    "fact_performance": scheme_perf,
    "SIP_inflows": sip_inflows,
}

for table_name, df in tables.items():

    db_count = pd.read_sql(
        f"SELECT COUNT(*) AS cnt FROM {table_name}",
        engine
    ).iloc[0]["cnt"]

    csv_count =len(df)

    print(f"\n{table_name}")
    print(f"CSV Rows : {csv_count}")
    print(f"DB Rows  : {db_count}")

    if csv_count == db_count:
        print("✓ Match")
    else:
        print("✗ Mismatch")



dim_fund
CSV Rows : 40
DB Rows  : 40
✓ Match

fact_nav
CSV Rows : 46000
DB Rows  : 46000
✓ Match

fact_aum
CSV Rows : 90
DB Rows  : 90
✓ Match

fact_transactions
CSV Rows : 32778
DB Rows  : 32778
✓ Match

fact_performance
CSV Rows : 40
DB Rows  : 40
✓ Match

SIP_inflows
CSV Rows : 48
DB Rows  : 48
✓ Match


In [19]:
# Normalise date columns to ISO YYYY-MM-DD

def to_iso(series):
    return pd.to_datetime(series, dayfirst=False, errors="coerce").dt.strftime("%Y-%m-%d")
 
nav_history["date"]            = to_iso(nav_history["date"])
aum_fh["date"]                 = to_iso(aum_fh["date"])
sip_inflows["month"]           = to_iso(sip_inflows["month"])
cat_inflows["month"]           = to_iso(cat_inflows["month"])
folio_count["month"]           = to_iso(folio_count["month"])
transactions["transaction_date"] = to_iso(transactions["transaction_date"])
holdings["portfolio_date"]     = to_iso(holdings["portfolio_date"])
benchmarks["date"]             = to_iso(benchmarks["date"])
fund_master["launch_date"]     = to_iso(fund_master["launch_date"])

In [21]:
# Collect ALL dates for dim_date
all_dates = pd.concat([
    nav_history["date"],
    aum_fh["date"],
    sip_inflows["month"],
    cat_inflows["month"],
    folio_count["month"],
    transactions["transaction_date"],
    holdings["portfolio_date"],
    benchmarks["date"],
    fund_master["launch_date"],
]).dropna()

dim_date_df = build_dim_date_df(all_dates)
print(f"dim_date rows to insert: {len(dim_date_df)}")


dim_date rows to insert: 1340


In [28]:
# ── 5. CONNECT & CREATE SCHEMA ────
import os

DB_PATH = r"C:\Users\orsc\Desktop\Database"
 
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
 
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(DDL)
conn.commit()
print("Schema created.")
 



PermissionError: [WinError 5] Access is denied: 'C:\\Users\\orsc\\Desktop\\Database'